# Buy or Bye — EDA ve Modelleme

UCI Online Shoppers verisi üzerinde yeniden üretilebilir analiz. Model tuning yalnız train setinde 5-fold CV PR-AUC ile yapılmış; calibration OOF tahminlerle kurulmuş ve test seti yalnız final raporlamada kullanılmıştır.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data import load_data, get_feature_groups
df = load_data(ROOT / 'data/raw/online_shoppers_intention.csv')
groups = get_feature_groups(df)
print(f'Satır: {len(df):,} | Sütun: {df.shape[1]} | Eksik: {df.isna().sum().sum()}')
df.head()

Satır: 12,205 | Sütun: 18 | Eksik: 0


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Month,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,1,1,1,1,Returning_Visitor,False,Feb,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,2,2,1,2,Returning_Visitor,False,Feb,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,4,1,9,3,Returning_Visitor,False,Feb,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,3,2,2,4,Returning_Visitor,False,Feb,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,3,3,1,4,Returning_Visitor,True,Feb,False


In [2]:
df['Revenue'].value_counts().rename(index={False: 'Bye', True: 'Buy'}).to_frame('Oturum')

,Oturum
Revenue,
Bye,10297
Buy,1908


## EDA görselleri

![Hedef dağılımı](../outputs/target_distribution.png)

![Korelasyon](../outputs/correlation_heatmap.png)

In [3]:
df[groups['numeric']].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
Administrative,12205.0,2.339,3.330,0.0,0.000,1.000,4.000,27.000
Administrative_Duration,12205.0,81.646,177.492,0.0,0.000,9.000,94.700,3398.750
Informational,12205.0,0.509,1.276,0.0,0.000,0.000,0.000,24.000
Informational_Duration,12205.0,34.825,141.425,0.0,0.000,0.000,0.000,2549.375
ProductRelated,12205.0,32.046,44.594,0.0,8.000,18.000,38.000,705.000
ProductRelated_Duration,12205.0,1206.982,1919.601,0.0,193.000,608.943,1477.155,63973.522
BounceRates,12205.0,0.020,0.045,0.0,0.000,0.003,0.017,0.200
ExitRates,12205.0,0.041,0.046,0.0,0.014,0.025,0.049,0.200
PageValues,12205.0,5.950,18.654,0.0,0.000,0.000,0.000,361.764
SpecialDay,12205.0,0.062,0.200,0.0,0.000,0.000,0.000,1.000


## Eğitim tasarımı

Aşağıdaki komut 70/15/15 stratified split, train üzerinde 5-fold tuning, OOF sigmoid calibration, maliyet-duyarlı threshold, ablation, multi-seed ve temporal proxy analizlerini çalıştırır.

```bash
python -m src.tune
```

In [4]:
selection = json.loads((ROOT / 'reports/feature_engineering_report.json').read_text())
rows = selection.get('tuned_engineered_results', selection.get('cv_results'))
pd.DataFrame(rows)[['model','cv_pr_auc_mean','cv_pr_auc_std','cv_train_pr_auc_mean']].round(4)

,model,cv_pr_auc_mean,cv_pr_auc_std,cv_train_pr_auc_mean
0,Engineered Random Forest,0.7529,0.0077,0.8837
1,Engineered LightGBM,0.7562,0.0080,0.8614


In [5]:
final_report = json.loads((ROOT / 'reports/ensemble_report.json').read_text())
test = final_report['test_metrics']
selected = 'RF + Engineered LightGBM Ensemble'
pd.Series(test)[['roc_auc','pr_auc','precision_optimized','recall_optimized','f1_optimized','threshold_optimized']].to_frame(f'{selected} test').round(4)

,RF + Engineered LightGBM Ensemble test
roc_auc,0.93198
pr_auc,0.744838
precision_optimized,0.587013
recall_optimized,0.79021
f1_optimized,0.673621
threshold_optimized,0.25


## Açıklanabilirlik

![SHAP summary](../outputs/shap_summary.png)

`PageValues` güçlü bir sinyal olsa da gerçek zamanlı kullanımda leakage riski ayrıca değerlendirilmelidir. SHAP ilişkisel model katkısını gösterir, nedensellik göstermez.